# Qwen3-8B-base-LoRA — extended data (passages_clean)

Same **LoRA24** recipe and hyperparameters as [`Qwen3-8B-base-Lora.ipynb`](Qwen3-8B-base-Lora.ipynb), but trained on **`extended_data/passages_clean.csv`** instead of host `train.csv`.

**5-fold prompt split** on extended `prompt_id`s (`FOLD_SEED = 0`). Training seed **7**. Two T4 workers. Output under `lora_out/` (`job_0`…`job_4` + `job_final`).

Submit with [`Qwen3-8B-base-Lora-submit.ipynb`](Qwen3-8B-base-Lora-submit.ipynb) using `job_final`.

## Attach on Kaggle
- Competition **Name That LLM** (optional for this train notebook; needed at submit)
- **Qwen3-8B-Base**
- Dataset containing `passages_clean.csv` (e.g. this repo’s `data/extended_data/`)
- **2× T4**, internet off

**Note:** Host-only scored **0.489** LB. Extended-only may not beat that on the real test (different register). See [`experiments.md`](experiments.md).


## Recipe

| Setting | Value |
| --- | --- |
| Base model | Qwen3-8B-Base truncated to **24** layers |
| LoRA | r=16, α=32, dropout=0.05 on `q,k,v,o,gate,up,down` |
| Pooling | mean pool (first token masked) over layers 2/4/6/8/12/16/20/24 |
| Heads | LayerNorm(4096) per layer → concat → plain + centered Linear |
| Microbatches | 3 same-prompt answers |
| Loss | CE(plain(z)) + CE(centered(z − microbatch mean)) |
| Optim | AdamW lr 1e-4 (adapters) / 1e-3 (heads+norms), wd 0.01 |
| Schedule | 10% warmup + cosine, grad clip 1.0, 3 epochs, accum 6 |
| Precision | fp16 frozen base, fp32 trainable, fp16 autocast + GradScaler(256), grad checkpointing |
| Seq length | max_len 447 + EOS |

**Jobs:** outer folds `0–4` (OOF features + plain/center readouts for held-out rows) and `final` (all rows; adapter saved for inference).


## Hyperparameters & imports


In [ ]:
SEED = 7
FOLD_SEED = 0
JOBS = {
    0: ['0', '2', '4'],       # GPU 0
    1: ['1', '3', 'final'],  # GPU 1
}
LAYERS = [2, 4, 6, 8, 12, 16, 20, 24]
KEEP = 24
MAX_LEN = 447
EPOCHS = 3
ACCUM = 6
LR_ADAPTER = 1e-4
LR_HEAD = 1e-3
WD = 0.01
WARMUP = 0.10
R = 16
ALPHA = 32
LDROP = 0.05

import glob
import json
import os
import pathlib
import subprocess
import sys
import time

import numpy as np
import pandas as pd


## Prepare extended CSV

Build `extended_train.csv` with columns `id, prompt_id, text, model`. Prefer `split == "new"` prompts that have all **18** models (best for triplets).


In [ ]:
def find_extended_clean():
    hits = []
    for root in (pathlib.Path('data/extended_data'), pathlib.Path('/kaggle/input'), pathlib.Path('.')):
        if not root.exists():
            continue
        if root.is_file():
            continue
        hits.extend(root.rglob('passages_clean.csv'))
    # dedupe
    uniq, seen = [], set()
    for h in hits:
        r = h.resolve()
        if r not in seen:
            seen.add(r)
            uniq.append(h)
    if not uniq:
        raise FileNotFoundError('passages_clean.csv not found — attach extended_data')
    return uniq[0]


ext_path = find_extended_clean()
ext = pd.read_csv(ext_path)
need = {'id', 'prompt_id', 'text', 'model'}
assert need <= set(ext.columns), need - set(ext.columns)

if 'split' in ext.columns:
    cand = ext[ext['split'] == 'new'].copy()
else:
    cand = ext.copy()
n_per = cand.groupby('prompt_id')['model'].nunique()
complete = set(n_per[n_per >= 18].index)
stage_df = cand[cand['prompt_id'].isin(complete)].copy()
if len(stage_df) < 1000:
    n_per = ext.groupby('prompt_id')['model'].nunique()
    ok = set(n_per[n_per >= 3].index)
    stage_df = ext[ext['prompt_id'].isin(ok)].copy()

stage_df = stage_df[['id', 'prompt_id', 'text', 'model']].dropna()
stage_df['text'] = stage_df['text'].astype(str)
stage_df.to_csv('extended_train.csv', index=False)
print('source', ext_path)
print('rows', len(stage_df), 'prompts', stage_df.prompt_id.nunique())
print(stage_df.model.value_counts().sort_index())


## Worker script

**Why `worker.py`?** The notebook is one process, but we have **two T4 GPUs** and want them training **different folds at the same time**. A single notebook cell can’t cleanly own two full independent trainings on two GPUs — they’d fight over memory/devices.

So we:
1. Write the training code into a normal file (`worker.py`) via the `%%writefile` / `%%writefile -a` cells below (run them in order).
2. Launch **two separate Python processes** later, each with its own `CUDA_VISIBLE_DEVICES` (`0` vs `1`) and its own job list (e.g. folds `0,2,4` vs `1,3,final`).

Each process = its own model in memory = its own GPU. That’s why training lives in `worker.py` instead of running only inside the notebook.


### 1. Imports, data, folds, tokenize

Creates `worker.py` (overwrites).


In [ ]:
%%writefile worker.py
import importlib.util as _iu
import json
import math
import os
import random
import sys
import time

import numpy as np
import pandas as pd
import torch
from safetensors.torch import save_file
from torch import nn
from transformers import AutoConfig, AutoModel, AutoTokenizer

# Kaggle ships torchao 0.10, which peft rejects; hide it before importing peft
_fs = _iu.find_spec
_iu.find_spec = lambda name, *a, **k: None if name.split('.')[0] == 'torchao' else _fs(name, *a, **k)

from peft import LoraConfig, get_peft_model
import peft.import_utils as _pi

_pi.is_torchao_available = lambda *a, **k: False

# --- config & jobs from the launcher ---
cfg = json.loads(sys.argv[1])
jobs = sys.argv[2].split(',')

# --- load train.csv ---
train = pd.read_csv(cfg['train_csv'])
LABELS = sorted(train.model.unique())
y = train.model.map({l: i for i, l in enumerate(LABELS)}).values
pid = train.prompt_id.values

# --- 5-fold prompt split (shuffle questions once, then bucket 0..4) ---
prompts = np.array(sorted(train.prompt_id.unique()))
rng0 = np.random.RandomState(cfg['fold_seed'])
rng0.shuffle(prompts)
fold = train.prompt_id.map({p: i % 5 for i, p in enumerate(prompts)}).values

# --- tokenize all passages once ---
tok = AutoTokenizer.from_pretrained(cfg['model_dir'])
tok.padding_side = 'right'
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

tokens = [
    ids + [tok.eos_token_id]
    for ids in tok(
        train.text.tolist(),
        add_special_tokens=False,
        truncation=True,
        max_length=cfg['max_len'],
    )['input_ids']
]

L = cfg['layers']  # which hidden layers to pool later
dev = torch.device('cuda:0')  # this process's only GPU (set via CUDA_VISIBLE_DEVICES)


### 2. Microbatch helper

Appends `triplets()` — builds training microbatches of **3 answers to the same question**. Needed because the centered head subtracts the mean of those 3 (same topic, different model styles).

**Detailed example**

Suppose the fit set has two prompts (after the fold split):

| row index | prompt | model (who wrote it) |
| --- | --- | --- |
| 10 | Q1 | gpt |
| 11 | Q1 | claude |
| 12 | Q1 | gemini |
| 13 | Q1 | kimi |
| 14 | Q1 | qwen |
| 20 | Q2 | gpt |
| 21 | Q2 | claude |
| 22 | Q2 | gemini |

**Step A — per prompt, shuffle answers, chunk by 3**

For Q1 (5 answers), after shuffle say order is `[10, 14, 11, 13, 12]`:

- chunk 1: `[10, 14, 11]` → already size 3 ✓
- leftover: `[13, 12]` → only 2, so **pad** with another Q1 row not already in the chunk, e.g. `10` → `[13, 12, 10]`

For Q2 (3 answers), after shuffle say `[22, 20, 21]`:

- one chunk: `[22, 20, 21]` ✓

**Step B — list of all triplets so far**

```text
[ [10, 14, 11], [13, 12, 10], [22, 20, 21] ]
```

**Step C — shuffle the order of triplets** (which microbatch comes first)

```text
[ [22, 20, 21], [10, 14, 11], [13, 12, 10] ]
```

**Step D — pad list length to a multiple of `ACCUM` (6)**

We only have 3 triplets; `3 % 6 != 0`, so duplicate random existing triplets until length is 6:

```text
[ [22, 20, 21], [10, 14, 11], [13, 12, 10],
  [10, 14, 11], [22, 20, 21], [13, 12, 10] ]  # last 3 are pads
```

Each inner list of 3 is one forward pass. Every `ACCUM=6` microbatches → one optimizer step (effective batch ≈ 3×6 = 18).


In [ ]:
%%writefile -a worker.py

def triplets(rows, rng):
    """Build microbatches of 3 answers to the same prompt; pad short chunks; pad to accum."""
    out = []
    for p in np.unique(pid[rows]):
        m = rng.permutation(rows[pid[rows] == p])
        for s in range(0, len(m), 3):
            b = list(m[s:s + 3])
            while len(b) < 3:  # fill a short last chunk with other answers to the same prompt
                b.append(int(rng.choice([x for x in m if x not in b])))
            out.append(b)
    out = [out[i] for i in rng.permutation(len(out))]
    while len(out) % cfg['accum']:
        out.append(out[rng.randint(len(out))])
    return out


### 3. Job setup: model, LoRA, heads, optimizer

Appends the start of the per-job loop.


In [ ]:
%%writefile -a worker.py

for job in jobs:
    t0 = time.time()
    outdir = os.path.join('lora_out', f'job_{job}')
    os.makedirs(outdir, exist_ok=True)

    random.seed(cfg['seed'])
    np.random.seed(cfg['seed'])
    torch.manual_seed(cfg['seed'])
    torch.cuda.manual_seed_all(cfg['seed'])
    rng = np.random.RandomState(cfg['seed'] + (99 if job == 'final' else int(job)))

    fit = np.arange(len(train)) if job == 'final' else np.where(fold != int(job))[0]
    valid = np.arange(3) if job == 'final' else np.where(fold == int(job))[0]

    mc = AutoConfig.from_pretrained(cfg['model_dir'])
    mc.num_hidden_layers, mc.use_cache = cfg['keep'], False
    if getattr(mc, 'layer_types', None) is not None:
        mc.layer_types = mc.layer_types[:cfg['keep']]

    base = AutoModel.from_pretrained(cfg['model_dir'], config=mc, dtype=torch.float16).to(dev)
    base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    base.enable_input_require_grads()
    model = get_peft_model(
        base,
        LoraConfig(
            r=cfg['r'],
            lora_alpha=cfg['alpha'],
            lora_dropout=cfg['ldrop'],
            bias='none',
            target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
        ),
    )
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()

    norms = nn.ModuleList([nn.LayerNorm(4096) for _ in L]).to(dev).float()
    plain = nn.Linear(4096 * len(L), len(LABELS)).to(dev).float()
    centered = nn.Linear(4096 * len(L), len(LABELS)).to(dev).float()
    drop = nn.Dropout(0.1)

    ad = [p for p in model.parameters() if p.requires_grad]
    hd = list(norms.parameters()) + list(plain.parameters()) + list(centered.parameters())
    opt = torch.optim.AdamW(
        [{'params': ad, 'lr': cfg['lr_adapter']}, {'params': hd, 'lr': cfg['lr_head']}],
        weight_decay=cfg['wd'],
    )
    plan = [triplets(fit, rng) for _ in range(cfg['epochs'])]
    total = sum(len(e) for e in plan) // cfg['accum']
    warm = max(1, int(cfg['warmup'] * total))
    sch = torch.optim.lr_scheduler.LambdaLR(
        opt,
        lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))),
    )
    scaler = torch.amp.GradScaler(init_scale=256.0)


### 4. Representation + training loop

Appends `rep()` and the CE train steps.


In [ ]:
%%writefile -a worker.py

    def rep(idx):
        batch = tok.pad({'input_ids': [tokens[i] for i in idx]}, return_tensors='pt').to(dev)
        with torch.autocast('cuda', dtype=torch.float16):
            hs = model(**batch, output_hidden_states=True, use_cache=False).hidden_states
        m = batch['attention_mask'].unsqueeze(-1).float()
        m[:, 0] = 0  # mask first token
        with torch.autocast('cuda', enabled=False):
            return torch.cat(
                [nm((hs[l].float() * m).sum(1) / m.sum(1)) for l, nm in zip(L, norms)],
                -1,
            )

    model.train()
    norms.train()
    step = 0
    skipped = 0
    losses = []
    for ep, batches in enumerate(plan):
        for i, b in enumerate(batches):
            z = rep(b)
            yy = torch.as_tensor(y[b], device=dev)
            loss = nn.functional.cross_entropy(plain(drop(z)), yy) + nn.functional.cross_entropy(
                centered(drop(z - z.mean(0, keepdim=True))), yy
            )
            scaler.scale(loss / cfg['accum']).backward()
            losses.append(float(loss.detach()))
            if (i + 1) % cfg['accum'] == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(ad + hd, 1.0)
                before = scaler.get_scale()
                scaler.step(opt)
                scaler.update()
                if scaler.get_scale() < before:
                    skipped += 1
                opt.zero_grad(set_to_none=True)
                sch.step()
                step += 1
                if step % 30 == 0:
                    print(
                        f'job {job} ep {ep} step {step}/{total} loss {np.mean(losses[-180:]):.4f} '
                        f'skipped {skipped} ({time.time() - t0:.0f}s)',
                        flush=True,
                    )


### 5. Eval + save artifacts

After training: fingerprint the held-out set, save artifacts under `lora_out/job_<id>/`, log a quick accuracy, free GPU memory. **No mid-training checkpoints** — one save per finished job after the fixed 3 epochs.

**OOF (out-of-fold)** = features/scores for a row from a model that **never trained on that row’s prompt**. Fold jobs produce that; stitching `job_0`…`job_4` gives an honest fingerprint for every train passage (good for stacking / blend weights / fair comparisons). Scoring train with `job_final` alone would leak (that model saw every prompt).

| Job | What we save | What it’s for |
| --- | --- | --- |
| `job_0` … `job_4` | `valid_features.npz` (OOF fingerprints) + `readouts.npz` (plain/center heads) + `COMPLETE.json` | Fair train-side features for ensembling later — **not** the main inference weights |
| `job_final` | same plus **`adapter/`**, **`norms.safetensors`**, **`tokenizer/`** | The model you load for real inference on new/test text |

**ELI5:** fold jobs = practice tests graded fairly on every homework question; `final` = the exam model you ship.


In [ ]:
%%writefile -a worker.py

    model.eval()
    norms.eval()
    Z = np.zeros((len(valid), 4096 * len(L)), np.float32)
    order = np.argsort([len(tokens[i]) for i in valid], kind='stable')
    with torch.inference_mode():
        for s in range(0, len(order), 6):
            pos = order[s:s + 6]
            Z[pos] = rep(valid[pos]).cpu().numpy()
    assert np.isfinite(Z).all()

    W = {n: t.weight.detach().cpu().numpy() for n, t in (('plain', plain), ('center', centered))}
    B = {n: t.bias.detach().cpu().numpy() for n, t in (('plain', plain), ('center', centered))}
    np.savez_compressed(
        os.path.join(outdir, 'readouts.npz'),
        classes=np.array(LABELS),
        pool_layers=np.array(L),
        plain_coef=W['plain'],
        plain_intercept=B['plain'],
        center_coef=W['center'],
        center_intercept=B['center'],
    )
    np.savez_compressed(
        os.path.join(outdir, 'valid_features.npz'),
        ids=train.id.values[valid],
        features=Z.astype(np.float16),
    )
    lg = Z @ W['plain'].T + B['plain']
    acc = float((lg.argmax(1) == y[valid]).mean())

    if job == 'final':
        model.save_pretrained(os.path.join(outdir, 'adapter'), safe_serialization=True)
        save_file(
            {**{f'norms.{k}': v.detach().cpu().contiguous() for k, v in norms.state_dict().items()}},
            os.path.join(outdir, 'norms.safetensors'),
        )
        tok.save_pretrained(os.path.join(outdir, 'tokenizer'))

    json.dump(
        {'job': job, 'valid_acc': acc, 'steps': step, 'skipped': skipped, 'secs': time.time() - t0},
        open(os.path.join(outdir, 'COMPLETE.json'), 'w'),
    )
    print(
        f'job {job} done: valid acc {acc:.4f} steps {step} skipped {skipped} ({time.time() - t0:.0f}s)',
        flush=True,
    )
    del model, base, opt, norms, plain, centered
    torch.cuda.empty_cache()


## Locate base model & launch workers

Uses `extended_train.csv` (from the prep cell). Finds Qwen3-8B-Base under `/kaggle/input`, then starts one process per GPU according to `JOBS`.


In [ ]:
T0 = time.time()
assert pathlib.Path('extended_train.csv').exists(), 'Run the prepare-extended cell first'
assert pathlib.Path('worker.py').exists(), 'Run the worker %%writefile cells first'

model_dir = None
search_globs = []
if pathlib.Path('/kaggle/input').is_dir():
    search_globs.append('/kaggle/input/**/config.json')
search_globs.append('**/config.json')
for pattern in search_globs:
    for c in sorted(glob.glob(pattern, recursive=True)):
        try:
            j = json.load(open(c))
        except Exception:
            continue
        if j.get('model_type') == 'qwen3' and j.get('hidden_size') == 4096 and j.get('num_hidden_layers') == 36:
            model_dir = os.path.dirname(c)
            break
    if model_dir:
        break
print('base', model_dir, flush=True)
assert model_dir

cfg = dict(
    train_csv='extended_train.csv',
    model_dir=model_dir,
    fold_seed=FOLD_SEED,
    seed=SEED,
    layers=LAYERS,
    keep=KEEP,
    max_len=MAX_LEN,
    epochs=EPOCHS,
    accum=ACCUM,
    lr_adapter=LR_ADAPTER,
    lr_head=LR_HEAD,
    wd=WD,
    warmup=WARMUP,
    r=R,
    alpha=ALPHA,
    ldrop=LDROP,
)

procs = []
for g, jl in JOBS.items():
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g), PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
    procs.append(
        (
            g,
            subprocess.Popen(
                [sys.executable, 'worker.py', json.dumps(cfg), ','.join(jl)],
                env=env,
                stdout=open(f'worker{g}.log', 'w'),
                stderr=subprocess.STDOUT,
            ),
        )
    )
for g, p in procs:
    p.wait()
    print(f'---- worker {g} (rc {p.returncode})')
    print(open(f'worker{g}.log').read()[-4000:])
done = sorted(glob.glob('lora_out/job_*/COMPLETE.json'))
summary = pd.DataFrame([json.load(open(d)) for d in done])
if not summary.empty:
    summary = summary[['job', 'valid_acc', 'steps', 'skipped', 'secs']]
    summary['valid_acc'] = summary['valid_acc'].map(lambda x: f'{x:.4f}')
    summary['secs'] = summary['secs'].map(lambda x: f'{x:.0f}')
print('completed jobs:')
print(summary.to_string(index=False))
print(f'total wall time: {time.time() - T0:.0f}s')
pd.DataFrame({'id': [], 'model': []}).to_parquet('submission.parquet')  # placeholder (training notebook)
